# ROR1–EGFR protein architecture across three breast-cancer proteomics cohorts

This notebook evaluates three prespecified associations in three published proteomics cohorts:

1. ROR1 abundance versus EGFR abundance;
2. the ROR1–EGFR pair-abundance score versus the MSigDB Hallmark EMT score;
3. the pair-abundance score versus a five-protein breast-cancer stem-cell (BCSC) marker score.

The analysis is Python-only and uses locally archived source supplements. Protein values are never imputed. Duplicate protein rows are collapsed to genes before programme scoring. The Anurag primary analysis is restricted to baseline biopsies; the mixed baseline/on-treatment analysis is retained only as an explicitly labelled sensitivity analysis.

The pair-abundance score is the mean of cohort-standardised ROR1 and EGFR abundance and is calculated only when both proteins were measured. It is an abundance summary—not evidence of molecular interaction or same-cell co-expression.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import openpyxl
import pandas as pd
import scipy
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [2]:
seed = 42
bootstrap_iterations = 10_000
permutation_iterations = 99_999
minimum_programme_coverage = 0.50
anurag_primary_event = "Baseline"

In [3]:
import os
os.environ["CAP_PROTEOME"] = "/protein-data"

In [4]:
rng = np.random.default_rng(seed)
sns.set_theme(style="whitegrid", context="notebook")

repository = next(
    candidate
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in (parent, parent / "publication_notebooks")
    if (candidate / "data" / "cancer_meta_programme_signatures.xlsx").is_file()
    and (candidate / "scripts" / "publication_figure_state.py").is_file()
)

In [5]:

if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
from scripts.publication_figure_state import project_paths, protein_paths


In [6]:
project, repository = project_paths()
notebook_dir = protein_paths(project)

FileNotFoundError: Set CAP_PROTEOME to the protein-data directory containing data/mmc2.xlsx.

In [ ]:

data_dir = notebook_dir / "data"
cache_dir = notebook_dir / "cache"
output_dir = notebook_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:

input_paths = {
    "lee_proteome": notebook_dir / "Lee_MOESM2_ESM.xlsx",
    "lee_subtypes": data_dir / "Lehmann.csv",
    "lee_uniprot_map": cache_dir / "lee_uniprot_gene_mapping.csv",
    "kosok_proteome": data_dir / "mmc2.xlsx",
    "anurag_proteome": data_dir / "cd-22-0200_table_s3_suppst3.xlsx",
    "anurag_metadata": data_dir / "cd-22-0200_table_s1_suppst1.xlsx",
    "msigdb_hallmarks": data_dir / "h.all.v2026.1.Hs.symbols.gmt",
}

missing_inputs = [str(path) for path in input_paths.values() if not path.exists()]
assert not missing_inputs, f"Missing inputs: {missing_inputs}"

print("Notebook directory resolved.")
print(f"Primary Anurag collection event: {anurag_primary_event}")
print(f"Seed: {seed}")

## Input provenance

SHA-256 hashes freeze the exact local files used by this execution. The UniProt mapping cache was generated by `scripts/validate_lee_sample_matching.py`; ambiguous accession-to-gene mappings are excluded below.

In [ ]:
def sha256(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


input_provenance = pd.DataFrame(
    [
        {
            "input": name,
            "relative_path": str(path.relative_to(notebook_dir)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for name, path in input_paths.items()
    ]
)
input_provenance.to_csv(output_dir / "three_proteome_input_provenance.csv", index=False)
input_provenance

## Signatures and helper functions

The EMT signature is read from the official MSigDB Hallmark GMT file. The BCSC score is a prespecified five-marker summary (`CD44`, `ALDH1A1`, `ALDH1A3`, `ITGA6`, `PROM1`) and must be described as such rather than as a comprehensive stemness programme.

For each cohort, duplicate measurements are collapsed by the median at gene level. Each protein is then z-standardised across the samples included in that analysis. A programme score is the mean of the available protein z-scores. Samples must contain at least 50% of the cohort-detectable signature proteins.

In [ ]:
emt_set_name = "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION"
bcsc_genes = ["CD44", "ALDH1A1", "ALDH1A3", "ITGA6", "PROM1"]


def read_gmt_set(path: Path, set_name: str) -> list[str]:
    with path.open() as handle:
        for line in handle:
            fields = line.rstrip("\n").split("\t")
            if fields and fields[0] == set_name:
                genes = [gene.strip() for gene in fields[2:] if gene.strip()]
                assert len(genes) == len(set(genes)), f"Duplicate genes in {set_name}"
                return genes
    raise KeyError(f"{set_name} was not found in {path}")


emt_genes = read_gmt_set(input_paths["msigdb_hallmarks"], emt_set_name)
assert len(emt_genes) == 200, f"Expected 200 Hallmark EMT genes, found {len(emt_genes)}"


def zscore_series(values: pd.Series) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce")
    standard_deviation = values.std(ddof=1)
    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        return pd.Series(np.nan, index=values.index, dtype=float)
    return (values - values.mean()) / standard_deviation


def collapse_gene_rows(frame: pd.DataFrame, gene_column: str) -> pd.DataFrame:
    numeric = frame.dropna(subset=[gene_column]).copy()
    numeric[gene_column] = numeric[gene_column].astype(str).str.strip()
    numeric = numeric.loc[numeric[gene_column].ne("") & numeric[gene_column].ne("na")]
    sample_columns = [column for column in numeric.columns if column != gene_column]
    numeric[sample_columns] = numeric[sample_columns].apply(pd.to_numeric, errors="coerce")
    return numeric.groupby(gene_column, sort=False)[sample_columns].median()


def programme_score(
    gene_by_sample: pd.DataFrame,
    genes: list[str],
    minimum_coverage: float = minimum_programme_coverage,
) -> tuple[pd.Series, pd.Series, list[str]]:
    genes_used = [gene for gene in genes if gene in gene_by_sample.index]
    assert genes_used, "No signature proteins were detected."
    values = gene_by_sample.loc[genes_used].astype(float)
    protein_z = values.sub(values.mean(axis=1), axis=0).div(
        values.std(axis=1, ddof=1).replace(0, np.nan), axis=0
    )
    observed_features = protein_z.notna().sum(axis=0)
    required_features = int(np.ceil(len(genes_used) * minimum_coverage))
    scores = protein_z.mean(axis=0, skipna=True)
    scores.loc[observed_features < required_features] = np.nan
    return scores, observed_features, genes_used


def assemble_scores(
    cohort: str,
    gene_by_sample: pd.DataFrame,
    subtype: pd.Series | None = None,
) -> tuple[pd.DataFrame, dict[str, object]]:
    missing_targets = sorted({"ROR1", "EGFR"} - set(gene_by_sample.index))
    assert not missing_targets, f"{cohort}: missing targets {missing_targets}"

    target_values = gene_by_sample.loc[["ROR1", "EGFR"]].T.copy()
    target_z = target_values.apply(zscore_series)
    pair_complete = target_z[["ROR1", "EGFR"]].notna().all(axis=1)
    pair_score = target_z.mean(axis=1, skipna=False).where(pair_complete)

    emt_score, emt_n, emt_used = programme_score(gene_by_sample, emt_genes)
    bcsc_score, bcsc_n, bcsc_used = programme_score(gene_by_sample, bcsc_genes)

    result = pd.DataFrame(
        {
            "cohort": cohort,
            "sample_id": gene_by_sample.columns,
            "ROR1": target_values["ROR1"],
            "EGFR": target_values["EGFR"],
            "ROR1_z": target_z["ROR1"],
            "EGFR_z": target_z["EGFR"],
            "pair_abundance_score": pair_score,
            "EMT_score": emt_score,
            "EMT_n_features": emt_n,
            "BCSC_score": bcsc_score,
            "BCSC_n_features": bcsc_n,
        }
    ).reset_index(drop=True)

    if subtype is not None:
        result["subtype"] = result["sample_id"].map(subtype)
    else:
        result["subtype"] = pd.NA

    audit = {
        "cohort": cohort,
        "samples_total": gene_by_sample.shape[1],
        "samples_with_both_targets": int(pair_complete.sum()),
        "emt_features_in_matrix": len(emt_used),
        "bcsc_features_in_matrix": len(bcsc_used),
        "emt_feature_symbols": ";".join(emt_used),
        "bcsc_feature_symbols": ";".join(bcsc_used),
    }
    return result, audit


print(f"Hallmark EMT genes: {len(emt_genes)}")
print(f"BCSC marker genes: {len(bcsc_genes)}")

## Lee cohort: 50 TNBC biopsies

In [ ]:
lee_raw = pd.read_excel(input_paths["lee_proteome"], sheet_name="Table S2", header=1)
lee_sample_columns = [
    column for column in lee_raw.columns if re.fullmatch(r"T\d+", str(column))
]
assert len(lee_sample_columns) == 50, f"Expected 50 Lee samples, found {len(lee_sample_columns)}"

uniprot_map = pd.read_csv(input_paths["lee_uniprot_map"]).dropna()
mapping_multiplicity = uniprot_map.groupby("uniprot_accession")["gene_name"].nunique()
unambiguous_accessions = set(mapping_multiplicity[mapping_multiplicity.eq(1)].index)
uniprot_to_gene = (
    uniprot_map.loc[uniprot_map["uniprot_accession"].isin(unambiguous_accessions)]
    .drop_duplicates("uniprot_accession")
    .set_index("uniprot_accession")["gene_name"]
    .to_dict()
)


def canonical_uniprot(accession: str) -> str:
    return re.sub(r"-\d+$", "", accession.strip().strip('"'))


def unambiguous_gene_for_protein_group(protein_group: object) -> object:
    accessions = {
        canonical_uniprot(accession)
        for accession in str(protein_group).split(";")
        if accession.strip()
    }
    genes = {uniprot_to_gene[accession] for accession in accessions if accession in uniprot_to_gene}
    return next(iter(genes)) if len(genes) == 1 else pd.NA


lee_raw["gene_symbol"] = lee_raw["ProteinID"].map(unambiguous_gene_for_protein_group)
lee_gene_matrix = collapse_gene_rows(
    lee_raw[["gene_symbol", *lee_sample_columns]], "gene_symbol"
)

lee_subtype_flags = pd.read_csv(input_paths["lee_subtypes"], sep=";", index_col=0)
lee_subtype_flags.columns = lee_subtype_flags.columns.str.strip()
lee_subtype_boolean = lee_subtype_flags.apply(
    lambda column: column.astype("string").str.strip().str.lower().eq("true")
)
assert lee_subtype_boolean.sum(axis=1).eq(1).all(), "Every Lee sample must have one subtype."
lee_subtypes = lee_subtype_boolean.idxmax(axis=1)
assert set(lee_sample_columns) == set(lee_subtypes.index), "Lee subtype/sample mismatch."

lee_scores, lee_audit = assemble_scores("Lee", lee_gene_matrix, lee_subtypes)
lee_scores.head()

## Kosok cohort: 18 breast-cancer cell lines

In [ ]:
kosok_raw = pd.read_excel(
    input_paths["kosok_proteome"],
    sheet_name="S1A median log2 fold change",
    skiprows=4,
)
kosok_prefix = "Normalized log2 fold-change: "
kosok_abundance_columns = [
    column for column in kosok_raw.columns if str(column).startswith(kosok_prefix)
]
kosok_renaming = {column: str(column).removeprefix(kosok_prefix) for column in kosok_abundance_columns}
kosok_gene_matrix = collapse_gene_rows(
    kosok_raw[["Gene", *kosok_abundance_columns]].rename(
        columns={"Gene": "gene_symbol", **kosok_renaming}
    ),
    "gene_symbol",
)
assert kosok_gene_matrix.shape[1] == 18, f"Expected 18 Kosok cell lines, found {kosok_gene_matrix.shape[1]}"

kosok_design = pd.read_excel(
    input_paths["kosok_proteome"],
    sheet_name="S1C experimental design",
    skiprows=2,
)
kosok_subtypes = (
    kosok_design.dropna(subset=["Cell Line"])
    .set_index("Cell Line")["This study"]
    .drop(index="Pooled Proteins", errors="ignore")
)
assert set(kosok_gene_matrix.columns) == set(kosok_subtypes.index), "Kosok subtype/sample mismatch."

kosok_scores, kosok_audit = assemble_scores("Kosok", kosok_gene_matrix, kosok_subtypes)
kosok_scores.head()

## Anurag cohort: baseline TNBC biopsies

The source workbook contains both baseline and Cycle 1 Day 3 specimens. The primary analysis below uses baseline specimens only and recalculates all z-scores within that baseline subset. The previous mixed-timepoint result is not described as a pretreatment analysis.

In [ ]:
anurag_raw = pd.read_excel(
    input_paths["anurag_proteome"],
    sheet_name="TableS3A",
    skiprows=2,
)
anurag_all_sample_columns = list(anurag_raw.columns[14:])

anurag_metadata = pd.read_excel(
    input_paths["anurag_metadata"],
    sheet_name="Table S1",
).set_index("sample_id", drop=False)

assert set(anurag_all_sample_columns).issubset(set(anurag_metadata.index)), (
    "Some Anurag proteomics columns are absent from Table S1."
)
anurag_primary_samples = [
    sample
    for sample in anurag_all_sample_columns
    if anurag_metadata.loc[sample, "Collection_Event"] == anurag_primary_event
]
assert len(anurag_primary_samples) == 55, (
    f"Expected 55 measured baseline samples, found {len(anurag_primary_samples)}"
)

anurag_gene_matrix = collapse_gene_rows(
    anurag_raw[["geneSymbol", *anurag_primary_samples]].rename(
        columns={"geneSymbol": "gene_symbol"}
    ),
    "gene_symbol",
)
anurag_scores, anurag_audit = assemble_scores("Anurag baseline", anurag_gene_matrix)
anurag_scores["tumour_content"] = pd.to_numeric(
    anurag_scores["sample_id"].map(anurag_metadata["TC"]), errors="coerce"
)

print(f"Proteomics samples in Table S3A: {len(anurag_all_sample_columns)}")
print(f"Measured baseline samples: {len(anurag_primary_samples)}")
print(f"Measured Cycle 1 Day 3 samples: {len(anurag_all_sample_columns) - len(anurag_primary_samples)}")
anurag_scores.head()

## Cohort and feature-coverage audit

In [ ]:
cohort_audit = pd.DataFrame([kosok_audit, lee_audit, anurag_audit])
cohort_audit.to_csv(output_dir / "three_proteome_cohort_audit.csv", index=False)

all_scores = pd.concat([kosok_scores, lee_scores, anurag_scores], ignore_index=True)
all_scores.to_csv(output_dir / "three_proteome_sample_scores.csv", index=False)

display_columns = [
    "cohort",
    "samples_total",
    "samples_with_both_targets",
    "emt_features_in_matrix",
    "bcsc_features_in_matrix",
]
cohort_audit[display_columns]

## Association tests

The sample is the replicate. Spearman correlations are accompanied by percentile bootstrap confidence intervals and fixed-seed Monte Carlo permutation p-values. Benjamini–Hochberg correction is applied jointly across the nine prespecified cohort–association tests. The permutation test preserves each variable's marginal distribution while breaking its sample pairing.

In [ ]:
association_definitions = {
    "ROR1–EGFR abundance": ("EGFR_z", "ROR1_z"),
    "Pair score–EMT": ("pair_abundance_score", "EMT_score"),
    "Pair score–BCSC": ("pair_abundance_score", "BCSC_score"),
}


def spearman_with_resampling(
    frame: pd.DataFrame,
    x_column: str,
    y_column: str,
    analysis_seed: int,
) -> dict[str, float | int]:
    complete = frame[[x_column, y_column]].replace([np.inf, -np.inf], np.nan).dropna()
    x = complete[x_column].to_numpy(float)
    y = complete[y_column].to_numpy(float)
    n = len(complete)
    assert n >= 4, f"Only {n} complete observations for {x_column} versus {y_column}"

    observed = stats.spearmanr(x, y)
    observed_rho = float(observed.statistic)
    analysis_rng = np.random.default_rng(analysis_seed)

    bootstrap_rho = np.empty(bootstrap_iterations, dtype=float)
    for iteration in range(bootstrap_iterations):
        indices = analysis_rng.integers(0, n, size=n)
        bootstrap_rho[iteration] = stats.spearmanr(x[indices], y[indices]).statistic
    finite_bootstrap = bootstrap_rho[np.isfinite(bootstrap_rho)]
    ci_low, ci_high = np.quantile(finite_bootstrap, [0.025, 0.975])

    x_rank = stats.rankdata(x)
    y_rank = stats.rankdata(y)
    x_rank = (x_rank - x_rank.mean()) / np.linalg.norm(x_rank - x_rank.mean())
    y_centred = y_rank - y_rank.mean()
    y_norm = np.linalg.norm(y_centred)
    observed_absolute = abs(observed_rho)
    exceedances = 0
    chunk_size = 2_000
    completed = 0
    while completed < permutation_iterations:
        current_size = min(chunk_size, permutation_iterations - completed)
        permuted = np.vstack([analysis_rng.permutation(y_centred) for _ in range(current_size)])
        permutation_rho = (permuted @ x_rank) / y_norm
        exceedances += int(np.count_nonzero(np.abs(permutation_rho) >= observed_absolute - 1e-15))
        completed += current_size
    permutation_p = (exceedances + 1) / (permutation_iterations + 1)

    return {
        "n": n,
        "rho": observed_rho,
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "asymptotic_p": float(observed.pvalue),
        "permutation_p": float(permutation_p),
        "bootstrap_finite": len(finite_bootstrap),
    }


cohort_frames = {
    "Kosok": kosok_scores,
    "Lee": lee_scores,
    "Anurag baseline": anurag_scores,
}

correlation_rows = []
analysis_number = 0
for cohort, frame in cohort_frames.items():
    for association, (x_column, y_column) in association_definitions.items():
        analysis_number += 1
        summary = spearman_with_resampling(
            frame,
            x_column,
            y_column,
            analysis_seed=seed + analysis_number,
        )
        correlation_rows.append(
            {
                "cohort": cohort,
                "association": association,
                "x_column": x_column,
                "y_column": y_column,
                **summary,
            }
        )

correlation_results = pd.DataFrame(correlation_rows)
correlation_results["fdr_bh"] = multipletests(
    correlation_results["permutation_p"], method="fdr_bh"
)[1]
correlation_results.to_csv(output_dir / "three_proteome_correlations.csv", index=False)
correlation_results

## AND-aligned and BCSC-marker sensitivity analyses

The mean pair-abundance score is retained as the primary summary because it matches the prespecified analysis. As a mechanistically conservative sensitivity analysis, the limiting-partner score is defined as `min(ROR1_z, EGFR_z)`. This score cannot establish the antigen-density threshold of the hemibody, but it prevents one highly abundant antigen from compensating for a low partner.

Because the BCSC score contains only five proteins, its robustness is also evaluated after excluding each marker in turn. These leave-one-marker-out results are descriptive stability checks, not five additional confirmatory hypothesis tests.

In [ ]:
limiting_partner_rows = []
for cohort_number, (cohort, frame) in enumerate(cohort_frames.items(), start=201):
    sensitivity_frame = frame.copy()
    sensitivity_frame["limiting_partner_score"] = sensitivity_frame[
        ["ROR1_z", "EGFR_z"]
    ].min(axis=1, skipna=False)
    for association_number, (association, y_column) in enumerate(
        [("Limiting partner–EMT", "EMT_score"), ("Limiting partner–BCSC", "BCSC_score")]
    ):
        limiting_partner_rows.append(
            {
                "cohort": cohort,
                "association": association,
                **spearman_with_resampling(
                    sensitivity_frame,
                    "limiting_partner_score",
                    y_column,
                    analysis_seed=seed + cohort_number + association_number,
                ),
            }
        )

limiting_partner_sensitivity = pd.DataFrame(limiting_partner_rows)
limiting_partner_sensitivity["fdr_bh"] = multipletests(
    limiting_partner_sensitivity["permutation_p"], method="fdr_bh"
)[1]
limiting_partner_sensitivity.to_csv(
    output_dir / "three_proteome_limiting_partner_sensitivity.csv", index=False
)

cohort_gene_matrices = {
    "Kosok": kosok_gene_matrix,
    "Lee": lee_gene_matrix,
    "Anurag baseline": anurag_gene_matrix,
}
bcsc_leave_one_out_rows = []
for cohort, frame in cohort_frames.items():
    gene_matrix = cohort_gene_matrices[cohort]
    pair_score = frame.set_index("sample_id")["pair_abundance_score"]
    for excluded_marker in bcsc_genes:
        retained_markers = [gene for gene in bcsc_genes if gene != excluded_marker]
        reduced_score, _, genes_used = programme_score(gene_matrix, retained_markers)
        complete = pd.concat(
            [pair_score.rename("pair_score"), reduced_score.rename("reduced_bcsc_score")],
            axis=1,
        ).dropna()
        result = stats.spearmanr(complete["pair_score"], complete["reduced_bcsc_score"])
        bcsc_leave_one_out_rows.append(
            {
                "cohort": cohort,
                "excluded_marker": excluded_marker,
                "retained_markers": ";".join(genes_used),
                "n": len(complete),
                "rho": result.statistic,
                "asymptotic_p_descriptive": result.pvalue,
            }
        )

bcsc_leave_one_out = pd.DataFrame(bcsc_leave_one_out_rows)
bcsc_leave_one_out.to_csv(
    output_dir / "three_proteome_bcsc_leave_one_out.csv", index=False
)

display(limiting_partner_sensitivity)
display(
    bcsc_leave_one_out.groupby("cohort")["rho"]
    .agg(["min", "max"])
    .rename(columns={"min": "leave_one_out_rho_min", "max": "leave_one_out_rho_max"})
)

## Anurag tumour-content sensitivity

This sensitivity analysis computes a partial Spearman correlation by rank-transforming the pair score, BCSC score and tumour content, then correlating the residuals after separate linear adjustment for tumour content. It does not replace the unadjusted primary association.

In [ ]:
def partial_spearman(frame: pd.DataFrame, x_column: str, y_column: str, z_column: str) -> pd.Series:
    complete = frame[[x_column, y_column, z_column]].replace([np.inf, -np.inf], np.nan).dropna()
    ranked = complete.rank(method="average")
    design = np.column_stack([np.ones(len(ranked)), ranked[z_column].to_numpy(float)])
    residual_x = ranked[x_column] - design @ np.linalg.lstsq(design, ranked[x_column], rcond=None)[0]
    residual_y = ranked[y_column] - design @ np.linalg.lstsq(design, ranked[y_column], rcond=None)[0]
    partial_rho = stats.pearsonr(residual_x, residual_y).statistic
    degrees_of_freedom = len(ranked) - 3
    test_statistic = partial_rho * np.sqrt(
        degrees_of_freedom / max(1e-15, 1 - partial_rho**2)
    )
    approximate_p = 2 * stats.t.sf(abs(test_statistic), degrees_of_freedom)
    return pd.Series(
        {
            "n": len(ranked),
            "partial_spearman_rho": partial_rho,
            "approximate_p": approximate_p,
        }
    )


anurag_tumour_content_sensitivity = partial_spearman(
    anurag_scores,
    "pair_abundance_score",
    "BCSC_score",
    "tumour_content",
).to_frame().T
anurag_tumour_content_sensitivity.to_csv(
    output_dir / "anurag_tumour_content_sensitivity.csv", index=False
)
anurag_tumour_content_sensitivity

## Publication figure

In [ ]:
cohort_order = ["Kosok", "Lee", "Anurag baseline"]
association_order = list(association_definitions)
scatter_axis_limits = {
    "ROR1–EGFR abundance": ((-2.5, 3.5), (-2.5, 2.5)),
    "Pair score–EMT": ((-2.0, 2.5), (-2.0, 1.5)),
    "Pair score–BCSC": ((-2.0, 2.5), (-1.5, 1.5)),
}
subtype_palettes = {
    "Kosok": {
        "Luminal": "#4477AA",
        "Basal A": "#EE6677",
        "Basal B": "#228833",
    },
    "Lee": dict(
        zip(
            sorted(lee_scores["subtype"].dropna().unique()),
            sns.color_palette("colorblind", lee_scores["subtype"].nunique()).as_hex(),
        )
    ),
}


def add_scatter_panel(
    axis: plt.Axes,
    frame: pd.DataFrame,
    cohort: str,
    association: str,
) -> None:
    x_column, y_column = association_definitions[association]
    complete = frame[[x_column, y_column, "subtype"]].dropna(subset=[x_column, y_column])

    if cohort in subtype_palettes:
        sns.scatterplot(
            data=complete,
            x=x_column,
            y=y_column,
            hue="subtype",
            palette=subtype_palettes[cohort],
            s=48,
            alpha=0.85,
            edgecolor="white",
            linewidth=0.4,
            legend=False,
            ax=axis,
        )
    else:
        sns.scatterplot(
            data=complete,
            x=x_column,
            y=y_column,
            color="#4477AA",
            s=48,
            alpha=0.85,
            edgecolor="white",
            linewidth=0.4,
            ax=axis,
        )

    result = correlation_results.loc[
        correlation_results["cohort"].eq(cohort)
        & correlation_results["association"].eq(association)
    ].iloc[0]
    axis.text(
        0.03,
        0.97,
        f"n={int(result['n'])}\nρ={result['rho']:.2f}\nPperm={result['permutation_p']:.3g}\nqBH={result['fdr_bh']:.3g}",
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )
    axis.axhline(0, color="#B0B0B0", linewidth=0.6, linestyle="--")
    axis.axvline(0, color="#B0B0B0", linewidth=0.6, linestyle="--")
    axis.set_title(association, fontsize=11)
    axis.set_xlabel(
        "EGFR abundance (z)" if association == "ROR1–EGFR abundance" else "ROR1–EGFR pair-abundance score"
    )
    axis.set_ylabel(
        "ROR1 abundance (z)"
        if association == "ROR1–EGFR abundance"
        else ("Hallmark EMT protein score" if association == "Pair score–EMT" else "Five-marker BCSC protein score")
    )
    # Corresponding panels use the same limits across cohorts.
    x_limits, y_limits = scatter_axis_limits[association]
    axis.set_xlim(*x_limits)
    axis.set_ylim(*y_limits)
    sns.despine(ax=axis)


figure, axes = plt.subplots(3, 3, figsize=(15, 13), constrained_layout=True)
for row, cohort in enumerate(cohort_order):
    frame = cohort_frames[cohort]
    for column, association in enumerate(association_order):
        add_scatter_panel(axes[row, column], frame, cohort, association)
        if column == 0:
            axes[row, column].text(
                -0.28,
                0.5,
                cohort,
                transform=axes[row, column].transAxes,
                rotation=90,
                ha="center",
                va="center",
                fontsize=13,
                fontweight="bold",
            )

for cohort, anchor_y in [("Kosok", 0.97), ("Lee", 0.64)]:
    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markerfacecolor=color,
            markeredgecolor="white",
            markersize=7,
            label=label,
        )
        for label, color in subtype_palettes[cohort].items()
    ]
    figure.legend(
        handles=handles,
        title=f"{cohort} subtype",
        loc="upper left",
        bbox_to_anchor=(1.005, anchor_y),
        frameon=False,
        fontsize=8,
        title_fontsize=9,
    )

figure.savefig(output_dir / "three_proteome_associations.png", dpi=300, bbox_inches="tight")
figure.savefig(output_dir / "three_proteome_associations.svg", bbox_inches="tight")
figure.savefig(output_dir / "three_proteome_associations.pdf", bbox_inches="tight")
plt.show()

## Correlation summary with bootstrap confidence intervals

In [ ]:
forest = correlation_results.copy()
forest["cohort"] = pd.Categorical(forest["cohort"], cohort_order, ordered=True)

figure, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=False, constrained_layout=True)
for axis, association in zip(axes, association_order):
    subset = forest.loc[forest["association"].eq(association)].sort_values("cohort")
    positions = np.arange(len(subset))
    axis.errorbar(
        subset["rho"],
        positions,
        xerr=np.vstack([subset["rho"] - subset["ci_low"], subset["ci_high"] - subset["rho"]]),
        fmt="o",
        color="#333333",
        ecolor="#666666",
        capsize=3,
    )
    axis.axvline(0, color="#B0B0B0", linewidth=0.8)
    axis.set_yticks(positions)
    if axis is axes[0]:
        axis.set_yticklabels(subset["cohort"])
    else:
        axis.set_yticklabels([])
    axis.set_xlim(-1, 1)
    axis.set_title(association, fontsize=10)
    axis.set_xlabel("Spearman ρ (95% bootstrap CI)")
    sns.despine(ax=axis)

figure.savefig(output_dir / "three_proteome_correlation_forest.png", dpi=300, bbox_inches="tight")
figure.savefig(output_dir / "three_proteome_correlation_forest.svg", bbox_inches="tight")
plt.show()

## Mixed-timepoint sensitivity analysis for Anurag

This section deliberately reproduces the earlier 71-column analysis as a sensitivity check. It contains 55 baseline and 16 Cycle 1 Day 3 specimens. Any result from this block must be labelled mixed-timepoint; it cannot be described as a pretreatment cohort.

In [ ]:
anurag_all_gene_matrix = collapse_gene_rows(
    anurag_raw[["geneSymbol", *anurag_all_sample_columns]].rename(
        columns={"geneSymbol": "gene_symbol"}
    ),
    "gene_symbol",
)
anurag_all_scores, anurag_all_audit = assemble_scores(
    "Anurag mixed timepoints", anurag_all_gene_matrix
)
anurag_all_scores["collection_event"] = anurag_all_scores["sample_id"].map(
    anurag_metadata["Collection_Event"]
)

mixed_rows = []
for analysis_number, (association, (x_column, y_column)) in enumerate(
    association_definitions.items(), start=101
):
    mixed_rows.append(
        {
            "cohort": "Anurag mixed timepoints",
            "association": association,
            **spearman_with_resampling(
                anurag_all_scores,
                x_column,
                y_column,
                analysis_seed=seed + analysis_number,
            ),
        }
    )

anurag_mixed_sensitivity = pd.DataFrame(mixed_rows)
anurag_mixed_sensitivity.to_csv(
    output_dir / "anurag_mixed_timepoint_sensitivity.csv", index=False
)
anurag_mixed_sensitivity

## Reproducibility record and final checks

In [ ]:
assert correlation_results.shape[0] == 9
assert set(correlation_results["n"]).issubset(set(range(4, 100)))
assert all_scores["sample_id"].notna().all()
assert not all_scores.duplicated(["cohort", "sample_id"]).any()
assert anurag_scores["sample_id"].map(anurag_metadata["Collection_Event"]).eq("Baseline").all()
assert int(anurag_all_audit["samples_with_both_targets"]) == 44
assert int(anurag_audit["samples_with_both_targets"]) == 34

software_versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "matplotlib": plt.matplotlib.__version__,
    "seaborn": sns.__version__,
    "openpyxl": openpyxl.__version__,
    "seed": seed,
    "bootstrap_iterations": bootstrap_iterations,
    "permutation_iterations": permutation_iterations,
    "minimum_programme_coverage": minimum_programme_coverage,
    "anurag_primary_event": anurag_primary_event,
}
(output_dir / "three_proteome_software_versions.json").write_text(
    json.dumps(software_versions, indent=2) + "\n"
)

print("All final checks passed.")
print(json.dumps(software_versions, indent=2))

Subtype sensitivity. The basal/TNBC subset is analysed both with the frozen full-panel scores and with scores recalculated within that subset. Subtype-adjusted associations in the full panel use within-subtype permutations. These are sensitivity analyses, not replacements selected for significance.

In [ ]:
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))

from scripts.publication_targeted_checks import protein_sensitivity

check_dir = repository / "results" / "publication_checks"
check_dir.mkdir(parents=True, exist_ok=True)
subtype_sensitivity = protein_sensitivity(repository, check_dir, project=project)
subtype_sensitivity

In [ ]:
import session_info

session_info.show()